# ⚽ 5.6 RLearnパッケージを用いたデータからの強化学習
World Cup 2022のトラッキング・イベントデータを用いて、データからの強化学習の実装と可視化を行います。

## 参考

*   オリジナルの手法[Nakahara+23]の論文：https://ieeexplore.ieee.org/abstract/document/10328596/
*   OpenSTARLab Preprocessing (code): https://github.com/open-starlab/PreProcessing
*   OpenSTARLab Preprocessing (SAR format doc): https://openstarlab.readthedocs.io/en/latest/Pre_Processing/Sports/SAR_data/index.html
*   OpenSTARLab RLearn (code): https://github.com/open-starlab/RLearn
*   OpenSTARLab RLearn (doc): https://openstarlab.readthedocs.io/en/latest/RLearn_Modeling/index.html

## クレジット
*   Authors: Kenjiro Ide & Keisuke Fujii
*   Affiliation: Nagoya University
*   Last updated: 2026-4-23
*   License: CC BY 4.0

強化学習においてGPUが必要なため、

ランタイム -> ランタイムのタイプを変更 -> T4 GPU

としてデフォルトのCPUからGPUに変更してください。

このノートブックで実行する手順は以下です。

1.  **環境構築**

2.  **FIFA World Cup 2022 データのダウンロード**

3.  **データの前処理**

4.  **データ分割とモデルの訓練**

5.  **Q値の可視化**

# 1.⚙️ 環境構築


*   Pythonのバージョンを3.10へ変更
*   Python3.10にpipをインストール
*   venvで仮想環境を構築
*   仮想環境のパスをシステムパスに設定
*   仮想環境へopenstarlabのパッケージをインストール



In [1]:
!python --version # Check python version

Python 3.12.13


In [2]:

# Install python 3.10
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-distutils -y

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,546 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,985 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-

Pathがpython3.10になっているSelectionの番号を入力

In [3]:
# Set default python version to python 3.10
# 1. Install Python 3.10 + pip quietly
!sudo apt-get update -qq > /dev/null 2>&1
!sudo apt-get install -y -qq python3.10 python3.10-distutils python3.10-venv curl > /dev/null 2>&1
!curl -sS https://bootstrap.pypa.io/get-pip.py | sudo python3.10 > /dev/null 2>&1

# 2. Set Python 3.10 as default quietly
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1 > /dev/null 2>&1
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.12 2 > /dev/null 2>&1
!sudo update-alternatives --set python3 /usr/bin/python3.10 > /dev/null 2>&1

# 3. Upgrade pip & install kernel quietly
!python3 -m pip install --upgrade pip setuptools wheel ipykernel -q > /dev/null 2>&1
!python3 -m ipykernel install --user --name=python310 --display-name "Python 3.10 (Colab)" > /dev/null 2>&1

# 4. Confirm python version 3.10.x
!python --version

Python 3.10.12


In [4]:
!python --version

Python 3.10.12


In [5]:
!pip --version

pip 26.0.1 from /usr/local/lib/python3.10/dist-packages/pip (python 3.10)


In [6]:
# Install pip module to python 3.10 environment
!wget https://bootstrap.pypa.io/get-pip.py
!python3.10 get-pip.py

--2026-04-25 23:46:20--  https://bootstrap.pypa.io/get-pip.py
Resolving bootstrap.pypa.io (bootstrap.pypa.io)... 151.101.0.175, 151.101.64.175, 151.101.128.175, ...
Connecting to bootstrap.pypa.io (bootstrap.pypa.io)|151.101.0.175|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2193439 (2.1M) [text/x-python]
Saving to: ‘get-pip.py’

get-pip.py          100%[===================>]   2.09M  --.-KB/s    in 0.04s   

2026-04-25 23:46:20 (49.7 MB/s) - ‘get-pip.py’ saved [2193439/2193439]

  Using cached pip-26.0.1-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1


In [7]:
!pip --version

pip 26.0.1 from /usr/local/lib/python3.10/dist-packages/pip (python 3.10)


In [8]:
!apt-get install -y python3.10-venv

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
python3.10-venv is already the newest version (3.10.12-1~22.04.15).
0 upgraded, 0 newly installed, 0 to remove and 107 not upgraded.


In [9]:
!python -m venv openstarlab_env

In [10]:
!/content/openstarlab_env/bin/python --version

Python 3.10.12


In [11]:
import os
import sys

venv_path = '/content/openstarlab_env'
python_version = "python3.10"
site_packages_path = os.path.join(venv_path, 'lib', python_version, 'site-packages')

if site_packages_path not in sys.path:
  sys.path.insert(0, site_packages_path)
  print(f"'{site_packages_path}' がシステムパスに追加されました。")
else:
  print("パスはすでに設定されています。")

# 現在のsys.pathの先頭が仮想環境になっているか確認
print("\n現在のsys.path[0]:", sys.path[0])

'/content/openstarlab_env/lib/python3.10/site-packages' がシステムパスに追加されました。

現在のsys.path[0]: /content/openstarlab_env/lib/python3.10/site-packages


In [12]:
# Install openstarlab module
!./openstarlab_env/bin/pip install \
    "numpy==1.24.4" \
    "scipy==1.10.1" \
    "pandas" \
    "openstarlab-preprocessing==0.1.43" \
    "openstarlab-rlearn==0.1.32"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 25.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 32.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 90.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.8/170.8 KB 28.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 KB 15.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 KB 13.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 110.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 KB 67.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.0/249.0 KB 42.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.0/472.0 KB 56.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 886.5/886.5 KB 82.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 12.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━

パッケージが正しくインストールされているか確認

In [13]:
import sys

# 1. Pythonがライブラリを探しに行くパスのリストを表示
print("--- sys.path ---")
for path in sys.path:
    print(path)

# 2. openstarlabが実際にインストールされている場所を確認
print("\n--- pip show openstarlab-preprocessing ---")
!/content/openstarlab_env/bin/python -m pip show openstarlab-preprocessing

print("\n--- pip show openstarlab-rlearn ---")
!/content/openstarlab_env/bin/python -m pip show openstarlab-rlearn

--- sys.path ---
/content/openstarlab_env/lib/python3.10/site-packages
/content
/env/python
/usr/lib/python312.zip
/usr/lib/python3.12
/usr/lib/python3.12/lib-dynload

/usr/local/lib/python3.12/dist-packages
/usr/lib/python3/dist-packages
/usr/local/lib/python3.12/dist-packages/IPython/extensions
/root/.ipython

--- pip show openstarlab-preprocessing ---
Name: openstarlab_preprocessing
Version: 0.1.43
Summary: openstarlab preprocessing package
Home-page: 
Author: 
Author-email: Calvin Yeung <yeung.chikwong@g.sp.m.is.nagoya-u.ac.jp>
License: Apache License 2.0
Location: /content/openstarlab_env/lib/python3.10/site-packages
Requires: ai2-tango, bidict, chardet, datasets, gdown, jsonlines, matplotlib, numpy, pandas, pyarrow, pydantic, scipy, shapely, statsbombpy, tqdm
Required-by: 

--- pip show openstarlab-rlearn ---
Name: openstarlab_rlearn
Version: 0.1.32
Summary: openstarlab rlearn modeliing package
Home-page: 
Author: 
Author-email: Kenjiro Ide <ide.kenjirog@g.sp.m.is.nagoya-u.ac.jp>

In [14]:
%%writefile check_library.py

# --- Matplotlibバックエンド設定（最重要） ---
# pyplotや、それを内部で使うライブラリ(preprocessing)をインポートする『前』に必ず記述します。
import matplotlib
matplotlib.use('Agg')
# -----------------------------------------

import sys
print("--- Python Script Start ---")
# 実行しているPythonのバージョンを確認
print(f"Executing with Python Version: {sys.version}")

try:
    # 正しいインポート文を記述
    import preprocessing
    import rlearn
    import numpy as np

    print("\n✅✅✅ openstarlab and numpy libraries were successfully imported!")
    print(f"NumPy Version: {np.__version__}")

except ImportError as e:
    print(f"\n❌❌❌ Import failed: {e}")

print("--- Python Script End ---")

Writing check_library.py


In [15]:
# Unset the environment variable and then run the script in one command
! unset MPLBACKEND && /content/openstarlab_env/bin/python check_library.py

--- Python Script Start ---
Executing with Python Version: 3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]

✅✅✅ openstarlab and numpy libraries were successfully imported!
NumPy Version: 1.24.4
--- Python Script End ---


# 2.📥 FIFA World Cup 2022のデータをダウンロード


PFF FCにより提供されたFIFA World Cup 2022のデータを使用します（
イベントデータ、トラッキングデータ、メタデータ、ロスターデータが含まれます）。


World Cup 2022データセットは、申請フォーム（https://www.blog.fc.pff.com/blog/pff-fc-release-2022-world-cup-data ）経由で無償提供されるものですが、公開ページおよび入手時点の通知において、2025年11月時点において明確な利用規約（再配布可否や商用利用の可否など）は提示されていませんでした。そのため、本書では当該データの再配布や同梱は行わず、各自で配布元のフォームから入手していただく方針とします。本Notebookは、手動でデータを配置する前提で提供しますので、入手後に各自の環境で設定して実行してください（利用範囲や掲示義務は将来変更される可能性があるため、最新の方針については配布元の案内を必ず確認し、不明点は配布元に問い合わせることを推奨します。なお、後日あらためて利用条件が提示された場合には、その条件に従って運用してください）。


In [16]:
# gdownライブラリをインストールします
!pip install gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [gdown]


In [17]:
# 必要なフォルダーを作成

!mkdir -p /content/data

!mkdir -p "/content/data/Rosters"
!mkdir -p "/content/data/Tracking Data"
!mkdir -p "/content/data/Event Data"
!mkdir -p "/content/data/Metadata"

ダウンロードしたイベントデータ（Event Data）、トラッキングデータ（Tracking Data）、メタデータ（Metadata）、ロスターデータ（Rosters）のフォルダの中にあるファイルを /content/data にセットしてください。あと、players.csvも /content/data  にセットする必要があります。

今回は、['3812', '3813', '3814', '3815', '3816']のファイルをそれぞれ4つの元フォルダから、左のファイルリストの /content/dataの各フォルダにドラッグ＆ドロップしてください。全試合をアップロードしようとすると、この方法では時間が掛かりそうなので、Google drive経由か、Google Colabを使わない方法を推奨します。

In [18]:
# ファイルを入れたあと、/content/data ディレクトリの中身を確認します
!ls -F /content/data/

'Event Data'/   Metadata/   players.csv   Rosters/  'Tracking Data'/


In [19]:
# Google Driveの共有リンクからファイルをダウンロードします
!gdown '1qzxkPE8j8AwapokYeOCCZb_Wc5j9ingE' -O 'config.zip'

!unzip config.zip

!gdown '1eFmKJBO-I3TToS0SA2AUClsV_hj4cRCZ' -O 'EPV_grid.csv'

Downloading...
From: https://drive.google.com/uc?id=1qzxkPE8j8AwapokYeOCCZb_Wc5j9ingE
To: /content/config.zip
100% 2.92k/2.92k [00:00<00:00, 16.2MB/s]
Archive:  config.zip
   creating: config/
  inflating: config/exp_fifawc2022.json  
  inflating: config/preprocessing_fifawc2022.json  
Downloading...
From: https://drive.google.com/uc?id=1eFmKJBO-I3TToS0SA2AUClsV_hj4cRCZ
To: /content/EPV_grid.csv
100% 11.2k/11.2k [00:00<00:00, 44.4MB/s]


# 3.✂️ openstarlab-preprocessingで前処理  

## 前処理

In [20]:
%%writefile preprocessing_fifawc2022.py

from pathlib import Path
from preprocessing import SAR_data

# Step 2: Define your data path as a Path object
data_directory = Path("/content/data")
config_path = Path("/content/config/preprocessing_fifawc2022.json")

# Step 3: Pass the Path object to the class
SAR_data(
    data_provider="fifawc",
    data_path=data_directory,
    state_def="PVS",
    config_path=config_path,
    preprocess_method="SAR",
    match_id=3812,
    max_workers=1
).preprocess_data()

SAR_data(
    data_provider="fifawc",
    data_path=data_directory,
    state_def="PVS",
    config_path=config_path,
    preprocess_method="SAR",
    match_id=3813,
    max_workers=1
).preprocess_data()

SAR_data(
    data_provider="fifawc",
    data_path=data_directory,
    state_def="PVS",
    config_path=config_path,
    preprocess_method="SAR",
    match_id=3814,
    max_workers=1
).preprocess_data()

print("\n✅ Preprocessing completed successfully!")

Writing preprocessing_fifawc2022.py


In [21]:
! unset MPLBACKEND && /content/openstarlab_env/bin/python preprocessing_fifawc2022.py

Starting data preprocessing...
Loading data from fifawc
  Removing 119 events with missing player/position data
  Protected 12 critical events from deletion
  Checking for halftime substitutions...
No halftime substitutions detected or all substitutions already recorded
  Saved play.csv with 1891 events (removed 119 incomplete events)
    Removed 19558 duplicate records before velocity calculation
    Applied velocity cap: 25.5 m/s, acceleration cap: ±763.5 m/s²
  Saved player.csv with 4067102 records (optimized processing)
    Removed 888 duplicate ball records
  Applied ball speed cap: 57.0 m/s
  Applied ball acceleration cap: ±2256.8 m/s²
  Saved ball.csv with 123162 records (optimized processing)
Processing 3812 finished
INFO:preprocessing.sports.SAR_data.soccer.soccer_SAR_cleaning:cleaning started... 3812
INFO:preprocessing.sports.SAR_data.soccer.cleaning.clean_tracking_data:Starting optimized player change detection...
INFO:preprocessing.sports.SAR_data.soccer.cleaning.clean_trac

In [22]:
%%writefile split_data_fifawc2022.py

from rlearn import RLearn_Model
from pathlib import Path

# デバッグのために、RLearn_Modelに渡される前にゲームIDが検出されるか確認します。
input_data_path = Path("/content/data/fifawc/preprocess_data/")

game_ids = []
if input_data_path.exists():
    for item in input_data_path.iterdir():
        if item.is_dir() and item.name.isdigit():
            game_ids.append(item.name)

print(f"Detected game IDs: {game_ids}")

# もしゲームIDが検出されていなければ、ここで処理を停止できます。
if not game_ids:
    print("エラー: 検出されたゲームIDがありません。データを分割できません。")
else:
    RLearn_Model(
        state_def="PVS",
        input_path=str(input_data_path),
        output_path="/content/data/fifawc/preprocess_data/split/",
    ).run_rlearn(
        run_split_train_test=True,
        run_preprocess_observation=False,
        run_train_and_test=False,
        run_visualize_data=False
    )

    print("\n✅ Spliting data completed successfully!")


Writing split_data_fifawc2022.py


In [23]:
! unset MPLBACKEND && /content/openstarlab_env/bin/python split_data_fifawc2022.py

Detected game IDs: ['3813', '3812', '3814']
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/json/json.py "HTTP/1.1 200 OK"
Setting num_proc from 4 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Generating train split: 142 examples [00:05, 25.94 examples/s]
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/json/json.py "HTTP/1.1 200 OK"
Setting num_proc from 4 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Generating train split: 164 examples [00:05, 28.85 examples/s]
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/json/json.py "HTTP/1.1 200 OK"
Setting num_proc from 4 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Generating train split: 139 examples [00:05, 24.26 examples/s]
Saving the dataset (1/1 shards): 100% 142/142 [

## 4. openstarlab-rlearnでデータを分割し、observationを作成し、学習と推論を実行します。

In [24]:
%%writefile observation_fifawc2022.py

from pathlib import Path
from rlearn import RLearn_Model
from rlearn.sports.main_class import PreprocessObservationConfig

config_path = Path("/content/config/preprocessing_fifawc2022.json")

# splitデータのパスを更新
split_data_base_path = "/content/data/fifawc/preprocess_data/split/"

RLearn_Model(
    state_def="PVS",
    config=config_path,
    num_process=1,
    input_path=split_data_base_path + "train",
    output_path="/content/data/fifawc/observation_data/split/train",
).run_rlearn(
    run_split_train_test=False,
    run_preprocess_observation=True,
    run_train_and_test=False,
    run_visualize_data=False,
    preprocess_config=PreprocessObservationConfig(batch_size=8)
)

RLearn_Model(
    state_def="PVS",
    config=config_path,
    num_process=1,
    input_path=split_data_base_path + "validation",
    output_path="/content/data/fifawc/observation_data/split/validation",
).run_rlearn(
    run_split_train_test=False,
    run_preprocess_observation=True,
    run_train_and_test=False,
    run_visualize_data=False,
    preprocess_config=PreprocessObservationConfig(batch_size=8)
)

RLearn_Model(
    state_def="PVS",
    config=config_path,
    num_process=1,
    input_path=split_data_base_path + "test",
    output_path="/content/data/fifawc/observation_data/split/test",
).run_rlearn(
    run_split_train_test=False,
    run_preprocess_observation=True,
    run_train_and_test=False,
    run_visualize_data=False,
    preprocess_config=PreprocessObservationConfig(batch_size=8)
)

print("\n✅ Creating Observation data completed successfully!")

Writing observation_fifawc2022.py


In [25]:
! unset MPLBACKEND && /content/openstarlab_env/bin/python observation_fifawc2022.py

INFO:root:input_path: /content/data/fifawc/preprocess_data/split/train
INFO:rlearn.sports.soccer.main_class_soccer.main:Length of dataset: 142
Map: 100% 142/142 [04:10<00:00,  1.77s/ examples]
INFO:rlearn.sports.soccer.main_class_soccer.main:Length of dataset after processing: 1420
Saving the dataset (2/2 shards): 100% 1420/1420 [00:06<00:00, 218.02 examples/s]
INFO:root:output_path: /content/data/fifawc/observation_data/split/train (elapsed: 257.46 sec)
INFO:root:input_path: /content/data/fifawc/preprocess_data/split/validation
INFO:rlearn.sports.soccer.main_class_soccer.main:Length of dataset: 164
Map: 100% 164/164 [04:14<00:00,  1.55s/ examples]
INFO:rlearn.sports.soccer.main_class_soccer.main:Length of dataset after processing: 1640
Saving the dataset (2/2 shards): 100% 1640/1640 [00:02<00:00, 771.71 examples/s]
INFO:root:output_path: /content/data/fifawc/observation_data/split/validation (elapsed: 257.01 sec)
INFO:root:input_path: /content/data/fifawc/preprocess_data/split/test
IN

In [26]:
import json

config_path = "/content/config/exp_fifawc2022.json"

with open(config_path, 'r') as f:
    config_data = json.load(f)

print(config_data)

# Modify some values in the config_data dictionary
config_data['dataset']['train_filename'] = '/content/data/fifawc/observation_data/split/train'
config_data['dataset']['valid_filename'] = '/content/data/fifawc/observation_data/split/validation'
config_data['dataset']['test_filename'] = '/content/data/fifawc/observation_data/split/test'

print(f"config: {config_data["dataset"]["test_filename"]}")

# changed learning rate and max epoch
print(f"max epochs: {config_data['max_epochs']}, learning rate: {config_data['model']['optimizer']['lr']}")

# Modify some values in the config_data dictionary
config_data['max_epochs'] = 1  # Example: Change max_epochs to 30
config_data['model']['optimizer']['lr'] = 0.01 # Example: Change learning rate to 0.001
config_data['datamodule']['type'] = 'rl_attacker' # Fix the datamodule type

# You can modify other values as needed
# config_data['datamodule']['batch_size'] = 128

# Print the modified values to confirm
print(f"max epochs: {config_data['max_epochs']}, learning rate: {config_data['model']['optimizer']['lr']}")
print(f"datamodule type: {config_data['datamodule']['type']}")

{'dataset': {'train_filename': '/content/data/fifawc/observation_data/split/train', 'valid_filename': '/content/data/fifawc/observation_data/split/validation', 'test_filename': '/content/data/fifawc/observation_data/split/test', 'preprocess_config': {'state_action_tokenizer': {'type': 'action_only', 'action2id': {'idle': 0, 'right': 1, 'up_right': 2, 'up': 3, 'up_left': 4, 'left': 5, 'down_left': 6, 'down': 7, 'down_right': 8, 'shot': 9, 'dribble': 10, 'through_pass': 11, 'pass': 12, 'cross': 13, 'defensive_action': 14}, 'origin_pos': 'center', 'special_tokens': ['[PAD]']}, 'num_workers': 8, 'preprocess_batch_size': 32}}, 'datamodule': {'type': 'jleague_rl_attacker', 'state_action_tokenizer': {'type': 'action_only', 'action2id': {'idle': 0, 'right': 1, 'up_right': 2, 'up': 3, 'up_left': 4, 'left': 5, 'down_left': 6, 'down': 7, 'down_right': 8, 'shot': 9, 'dribble': 10, 'through_pass': 11, 'pass': 12, 'cross': 13, 'defensive_action': 14}, 'origin_pos': 'center', 'special_tokens': ['[PAD

In [27]:
! unset MPLBACKEND && /content/openstarlab_env/bin/python train_fifawc2022.py

/content/openstarlab_env/bin/python: can't open file '/content/train_fifawc2022.py': [Errno 2] No such file or directory


In [28]:
with open(config_path, 'w') as f:
    json.dump(config_data, f, indent=4)

print(f"Modified config saved to {config_path}")

Modified config saved to /content/config/exp_fifawc2022.json


GPUに変更するときに、ランタイムがリセットされるため、作成したファイルをzip圧縮してGoogle Driveへ保存

In [30]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [31]:
# config, data, openstarlab_env フォルダと、直下にある python ファイルなどをまとめて固める
!zip -r /content/drive/MyDrive/colab_data_backup.zip data config openstarlab_env *.py *.csv

ストリーミング出力は最後の 5000 行に切り捨てられました。
  adding: openstarlab_env/lib/python3.10/site-packages/numpy/distutils/extension.py (deflated 72%)
  adding: openstarlab_env/lib/python3.10/site-packages/numpy/distutils/msvc9compiler.py (deflated 61%)
  adding: openstarlab_env/lib/python3.10/site-packages/numpy/distutils/system_info.py (deflated 78%)
  adding: openstarlab_env/lib/python3.10/site-packages/numpy/distutils/numpy_distribution.py (deflated 51%)
  adding: openstarlab_env/lib/python3.10/site-packages/numpy/distutils/ccompiler_opt.py (deflated 77%)
  adding: openstarlab_env/lib/python3.10/site-packages/numpy/distutils/line_endings.py (deflated 74%)
  adding: openstarlab_env/lib/python3.10/site-packages/numpy/distutils/exec_command.py (deflated 66%)
  adding: openstarlab_env/lib/python3.10/site-packages/numpy/distutils/command/ (stored 0%)
  adding: openstarlab_env/lib/python3.10/site-packages/numpy/distutils/command/install.py (deflated 63%)
  adding: openstarlab_env/lib/python3.10/site-package

## 学習・評価
ここから強化学習のためにGPUを使用するため、CPUからGPUに変更する必要がある。

手順

ランタイム -> ランタイムのタイプを変更 -> T4 GPU

Google DriveからGoogle Colabへフォルダをアップロード

In [32]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
!cp /content/drive/MyDrive/colab_data_backup.zip /content/

In [34]:
!unzip /content/colab_data_backup.zip -d /content/

ストリーミング出力は最後の 5000 行に切り捨てられました。
  inflating: /content/openstarlab_env/lib/python3.10/site-packages/numpy/array_api/tests/test_data_type_functions.py  
  inflating: /content/openstarlab_env/lib/python3.10/site-packages/numpy/array_api/__init__.py  
  inflating: /content/openstarlab_env/lib/python3.10/site-packages/numpy/array_api/_constants.py  
  inflating: /content/openstarlab_env/lib/python3.10/site-packages/numpy/array_api/__pycache__/linalg.cpython-310.pyc  
  inflating: /content/openstarlab_env/lib/python3.10/site-packages/numpy/array_api/__pycache__/setup.cpython-310.pyc  
  inflating: /content/openstarlab_env/lib/python3.10/site-packages/numpy/array_api/__pycache__/_constants.cpython-310.pyc  
  inflating: /content/openstarlab_env/lib/python3.10/site-packages/numpy/array_api/__pycache__/_sorting_functions.cpython-310.pyc  
  inflating: /content/openstarlab_env/lib/python3.10/site-packages/numpy/array_api/__pycache__/_dtypes.cpython-310.pyc  
  inflating: /content/openstarlab_env

class_weightsのみ計算を行い、ファイルを保存する

In [35]:
%%writefile train_fifawc2022_class_weight_only.py
import json
from pathlib import Path

from rlearn import RLearn_Model
from rlearn.sports.main_class import TrainAndTestConfig

base_config_path = Path("/content/config/exp_fifawc2022.json")
cached_config_path = Path("/content/config/exp_fifawc2022_with_cached_class_weight.json")
class_weight_cache_path = Path("/content/output/class_weight/exp_fifawc2022_class_weights.pt")

config = json.loads(base_config_path.read_text(encoding="utf-8"))
if "class_weight_fn" not in config:
    raise ValueError(
        "`/content/config/exp_fifawc2022.json` に `class_weight_fn` がありません。"
        "先に `class_weight_fn.type` などを設定してください。"
    )

# Set num_workers to 0 to avoid DataLoader worker issues
config["datamodule"]["num_workers"] = 0

config["class_weight_fn"]["cache_path"] = str(class_weight_cache_path)

cached_config_path.parent.mkdir(parents=True, exist_ok=True)
cached_config_path.write_text(
    json.dumps(config, ensure_ascii=False, indent=4),
    encoding="utf-8",
)

RLearn_Model(
    state_def="PVS"
).run_rlearn(
    run_split_train_test=False,
    run_preprocess_observation=False,
    run_train_and_test=True,
    run_visualize_data=False,
    train_and_test_config=TrainAndTestConfig(
        exp_name="sarsa_attacker",
        run_name="test",
        accelerator="gpu",
        devices=1,
        strategy="auto",
        exp_config_path=str(cached_config_path),
        use_class_weights=True,
        output_base_dir="/content/output",
        class_weight_only=True,
    ),
)

print(f"\n✅ Class weights saved to: {class_weight_cache_path}")
print("✅ Class-weight-only step completed successfully!")

Writing train_fifawc2022_class_weight_only.py


In [36]:
! unset MPLBACKEND && /content/openstarlab_env/bin/python train_fifawc2022_class_weight_only.py

Seed set to 42
INFO:rlearn.sports.soccer.main_class_soccer.main:loading dataset...
INFO:rlearn.sports.soccer.main_class_soccer.main:Preprocessing dataset...
Map (num_proc=8): 100% 1420/1420 [04:32<00:00,  5.22 examples/s]
INFO:rlearn.sports.soccer.main_class_soccer.main:Preprocessing dataset is done. 272.2043888568878 sec
INFO:rlearn.sports.soccer.main_class_soccer.main:Train dataset size: 1420
INFO:rlearn.sports.soccer.main_class_soccer.main:Prepare class weights...
INFO:rlearn.sports.soccer.class_weight.class_weight:No cache found at /content/output/class_weight/exp_fifawc2022_class_weights.pt
INFO:rlearn.sports.soccer.main_class_soccer.main:Calculating class weights...
calculating class weights: 100% 3/3 [03:45<00:00, 75.04s/it]
INFO:rlearn.sports.soccer.class_weight.class_weight:Saving class weights to /content/output/class_weight/exp_fifawc2022_class_weights.pt
INFO:rlearn.sports.soccer.main_class_soccer.main:Prepare class weights is done. 225.18345379829407 sec
INFO:rlearn.sports

計算済みのclass_weightsファイルを使用し学習を行う

In [37]:
%%writefile train_fifawc2022_with_cached_class_weight.py
import json
from pathlib import Path

from rlearn import RLearn_Model
from rlearn.sports.main_class import TrainAndTestConfig

base_config_path = Path("/content/config/exp_fifawc2022.json")
cached_config_path = Path("/content/config/exp_fifawc2022_with_cached_class_weight.json")
class_weight_cache_path = Path("/content/output/class_weight/exp_fifawc2022_class_weights.pt")

config = json.loads(base_config_path.read_text(encoding="utf-8"))
if "class_weight_fn" not in config:
    raise ValueError(
        "`/content/config/exp_fifawc2022.json` に `class_weight_fn` がありません。"
        "先に `class_weight_fn.type` などを設定してください。"
    )

# Set num_workers to 0 to avoid DataLoader worker issues
config["datamodule"]["num_workers"] = 0

config["class_weight_fn"]["cache_path"] = str(class_weight_cache_path)

cached_config_path.parent.mkdir(parents=True, exist_ok=True)
cached_config_path.write_text(
    json.dumps(config, ensure_ascii=False, indent=4),
    encoding="utf-8",
)

if not class_weight_cache_path.exists():
    raise FileNotFoundError(
        f"Cached class weights not found: {class_weight_cache_path}\n"
        "先に `train_fifawc2022_class_weight_only.py` を実行してください。"
    )

RLearn_Model(
    state_def="PVS"
).run_rlearn(
    run_split_train_test=False,
    run_preprocess_observation=False,
    run_train_and_test=True,
    run_visualize_data=False,
    train_and_test_config=TrainAndTestConfig(
        exp_name="sarsa_attacker",
        run_name="test",
        accelerator="gpu",
        devices=1,
        strategy="auto",
        exp_config_path=str(cached_config_path),
        use_class_weights=True,
        output_base_dir="/content/output",
        class_weight_only=False,
        save_q_values_csv=True,
        max_games_csv=1,
        max_sequences_per_game_csv=5,
    ),
)

print("\n✅ Training completed successfully with cached class weights!")

Writing train_fifawc2022_with_cached_class_weight.py


In [38]:
! unset MPLBACKEND && /content/openstarlab_env/bin/python train_fifawc2022_with_cached_class_weight.py

Seed set to 42
INFO:rlearn.sports.soccer.main_class_soccer.main:loading dataset...
INFO:rlearn.sports.soccer.main_class_soccer.main:Preprocessing dataset...
Map (num_proc=8): 100% 1420/1420 [05:02<00:00,  4.70 examples/s]
Map (num_proc=8): 100% 1640/1640 [05:13<00:00,  5.22 examples/s]
Map (num_proc=8): 100% 1390/1390 [04:51<00:00,  4.77 examples/s]
INFO:rlearn.sports.soccer.main_class_soccer.main:Preprocessing dataset is done. 907.8145174980164 sec
INFO:rlearn.sports.soccer.main_class_soccer.main:Train dataset size: 1420
INFO:rlearn.sports.soccer.main_class_soccer.main:Valid dataset size: 1640
INFO:rlearn.sports.soccer.main_class_soccer.main:Test dataset size: 1390
INFO:rlearn.sports.soccer.main_class_soccer.main:Prepare class weights...
INFO:rlearn.sports.soccer.class_weight.class_weight:Loading class weights from /content/output/class_weight/exp_fifawc2022_class_weights.pt
INFO:rlearn.sports.soccer.main_class_soccer.main:Prepare class weights is done. 0.03232312202453613 sec
INFO:rl

# 📄 5.Q値を可視化します。

In [39]:
!./openstarlab_env/bin/pip install \
    "adjustText" \
    "pykakasi"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 KB 19.9 MB/s eta 0:00:00


In [40]:
import json

config_path = "/content/config/exp_fifawc2022.json"

with open(config_path, 'r') as f:
    config_data = json.load(f)

print(config_data)
# Modify some values in the config_data dictionary
config_data['dataset']['test_filename'] = '/data/fifawc/observation_data/split/test'

print(f"config: {config_data["dataset"]["test_filename"]}")

with open(config_path, 'w') as f:
    json.dump(config_data, f, indent=4)

print(f"✅ Config updated and saved. Path set to: {config_data['dataset']['test_filename']}")

{'dataset': {'train_filename': '/content/data/fifawc/observation_data/split/train', 'valid_filename': '/content/data/fifawc/observation_data/split/validation', 'test_filename': '/content/data/fifawc/observation_data/split/test', 'preprocess_config': {'state_action_tokenizer': {'type': 'action_only', 'action2id': {'idle': 0, 'right': 1, 'up_right': 2, 'up': 3, 'up_left': 4, 'left': 5, 'down_left': 6, 'down': 7, 'down_right': 8, 'shot': 9, 'dribble': 10, 'through_pass': 11, 'pass': 12, 'cross': 13, 'defensive_action': 14}, 'origin_pos': 'center', 'special_tokens': ['[PAD]']}, 'num_workers': 8, 'preprocess_batch_size': 32}}, 'datamodule': {'type': 'rl_attacker', 'state_action_tokenizer': {'type': 'action_only', 'action2id': {'idle': 0, 'right': 1, 'up_right': 2, 'up': 3, 'up_left': 4, 'left': 5, 'down_left': 6, 'down': 7, 'down_right': 8, 'shot': 9, 'dribble': 10, 'through_pass': 11, 'pass': 12, 'cross': 13, 'defensive_action': 14}, 'origin_pos': 'center', 'special_tokens': ['[PAD]']}, 'n

In [48]:
# /content/output/ 配下のファイルを openstarlab_env 内に移動
!mkdir -p /content/output/figures/exp_config/players_q_state
!mkdir -p /content/openstarlab_env/lib/python3.10/site-packages/rlearn/sports/output/figures/exp_config/
!cp -r /content/output/figures/exp_config/players_q_state /content/openstarlab_env/lib/python3.10/site-packages/rlearn/sports/output/figures/exp_config/

# 確認
!ls -R /content/openstarlab_env/lib/python3.10/site-packages/rlearn/sports/output/figures/exp_config/

/content/openstarlab_env/lib/python3.10/site-packages/rlearn/sports/output/figures/exp_config/:
players_q_state

/content/openstarlab_env/lib/python3.10/site-packages/rlearn/sports/output/figures/exp_config/players_q_state:


In [49]:
%%writefile visualize_qvalues.py
from pathlib import Path
import os
from rlearn import RLearn_Model
from rlearn.sports.main_class import VisualizeDataConfig

config_path = Path("/content/config/exp_fifawc2022.json")

RLearn_Model(
    state_def="PVS"
).run_rlearn(
    run_split_train_test=False,
    run_preprocess_observation=False,
    run_train_and_test=False,
    run_visualize_data=True,
    visualize_config=VisualizeDataConfig(
        exp_config_path=config_path,
        checkpoint_path="/content/output/sarsa_attacker/test/checkpoints/best-00-6-val_loss=0.1940.ckpt",
        model_name="exp_config",
        tracking_file_path="/content/data/fifawc/preprocess_data/3812/events.jsonl",
        match_id="3814",
        sequence_id=0,
        viz_style="bar",
        movie_output_dir="/content/output",
        keep_frames=True
    ),
)

print("\n✅ Updated visualize_qvalues.py without unsupported argument!")

Overwriting visualize_qvalues.py


In [50]:
! unset MPLBACKEND && /content/openstarlab_env/bin/python visualize_qvalues.py

Map (num_proc=8): 100% 1390/1390 [04:27<00:00,  5.20 examples/s]
start loading 3814 0
100% 1390/1390 [05:58<00:00,  3.87it/s]
Error loading data: [Errno 2] No such file or directory: '/content/openstarlab_env/lib/python3.10/site-packages/rlearn/sports/output/figures/exp_config/players_q_state/q_values_3814_0.csv'

✅ Updated visualize_qvalues.py without unsupported argument!


In [45]:
from IPython.display import Image
Image('/content/output/frames/3814_0_Homam Ahmed/frame_0365.png')

FileNotFoundError: No such file or directory: '/content/output/frames/3814_0_Homam Ahmed/frame_0365.png'

FileNotFoundError: No such file or directory: '/content/output/frames/3814_0_Homam Ahmed/frame_0365.png'

<IPython.core.display.Image object>